In [1]:
import time
import re
import requests
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

In [13]:
# ============================================================
# Basic Helpers
# ============================================================

def clean_join(items):
    """
    Clean a list and return comma-separated string.
    """
    if not items:
        return np.nan

    cleaned = []

    for x in items:
        if x is None:
            continue

        s = str(x).strip()

        if not s:
            continue

        if s.lower() in {"nan", "none", "null"}:
            continue

        if s not in cleaned:
            cleaned.append(s)

    return ", ".join(cleaned) if cleaned else np.nan


def normalize_doi(doi_val):
    """
    Convert DOI URL to standard format.
    """
    if not isinstance(doi_val, str):
        return np.nan

    doi_val = doi_val.strip()

    if not doi_val:
        return np.nan

    doi_val = re.sub(
        r"^https?://(dx\.)?doi\.org/",
        "",
        doi_val,
        flags=re.IGNORECASE
    )

    return doi_val if doi_val else np.nan


def reconstruct_abstract(inv_idx):
    """
    OpenAlex abstracts are stored as inverted index.
    Reconstruct readable text.
    """
    if not inv_idx:
        return np.nan

    try:
        max_pos = max(max(lst) for lst in inv_idx.values())
    except Exception:
        return np.nan

    arr = [""] * (max_pos + 1)

    for word, positions in inv_idx.items():
        for p in positions:
            if 0 <= p < len(arr):
                arr[p] = word

    text = " ".join(arr)

    return text if text.strip() else np.nan


# ============================================================
# OpenAlex Label Extraction
# ============================================================

def extract_topic_parts(topic):
    """
    Extract hierarchical labels from OpenAlex topic.
    """
    if not isinstance(topic, dict):
        return None, None, None, None

    topic_name = topic.get("display_name")

    subfield = topic.get("subfield", {}).get("display_name")
    field = topic.get("field", {}).get("display_name")
    domain = topic.get("domain", {}).get("display_name")

    return topic_name, subfield, field, domain


def extract_labels(work):
    """
    Extract hierarchical labels from OpenAlex work.
    """
    primary_topic = work.get("primary_topic")
    topics = work.get("topics", []) or []

    topic_names = []
    subfields = []
    fields = []
    domains = []

    all_topics = []

    if isinstance(primary_topic, dict):
        all_topics.append(primary_topic)

    for t in topics:
        if isinstance(t, dict):
            all_topics.append(t)

    for t in all_topics:
        topic_name, subfield, field, domain = extract_topic_parts(t)

        if topic_name:
            topic_names.append(topic_name)

        if subfield:
            subfields.append(subfield)

        if field:
            fields.append(field)

        if domain:
            domains.append(domain)

    primary_name, primary_subfield, primary_field, primary_domain = (
        extract_topic_parts(primary_topic)
    )

    return {
        "domains": clean_join(domains),
        "fields": clean_join(fields),
        "subfields": clean_join(subfields),
        "topics": clean_join(topic_names),

        # Primary labels (best for classification)
        "primary_domain": primary_domain,
        "primary_field": primary_field,
        "primary_subfield": primary_subfield,
        "primary_topic": primary_name,
    }


# ============================================================
# OpenAlex Search
# ============================================================

def search_openalex(term, limit):
    """
    Query OpenAlex API for works.
    """
    url = "https://api.openalex.org/works"

    r = requests.get(
        url,
        params={
            "search": term,
            "per-page": limit,
            "mailto": "you@example.com",
        },
        timeout=30
    )

    if r.status_code != 200:
        print(f"OpenAlex error {r.status_code}")
        return []

    results = []

    for w in r.json().get("results", []):

        authors = [
            a.get("author", {}).get("display_name")
            for a in w.get("authorships", [])
        ]

        doi = normalize_doi(
            w.get("doi")
        )

        url_out = (
            w.get("primary_location", {})
            .get("landing_page_url")
        )

        abstract = reconstruct_abstract(
            w.get("abstract_inverted_index")
        )

        labels = extract_labels(w)

        results.append({

            "search_term": term,

            "doi": doi,

            "url": url_out,

            "title": w.get("title"),

            "authors": clean_join(authors),

            "abstract": abstract,

            **labels

        })

    return results


# ============================================================
# Main Pipeline
# ============================================================

def run_all_searches(
    search_terms,
    limit=10,
    min_words=25
):
    """
    Run OpenAlex searches and return cleaned dataframe.
    """

    all_results = []

    for i, term in enumerate(search_terms):

        print(f"Searching: {term} ({i+1}/{len(search_terms)})")

        res = search_openalex(
            term,
            limit
        )

        print(f"  Found {len(res)} results")

        all_results.extend(res)

        time.sleep(0.2)

    df = pd.DataFrame(all_results)

    if df.empty:
        return df

    # --------------------------------------------------------
    # Clean whitespace
    # --------------------------------------------------------

    for col in df.select_dtypes(include="object"):
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    df = df.replace(
        ["nan", "None", "null", ""],
        np.nan
    )

    # --------------------------------------------------------
    # Remove missing core fields
    # --------------------------------------------------------

    df = df.dropna(
        subset=["title", "abstract"]
    )

    # --------------------------------------------------------
    # Remove very short abstracts
    # --------------------------------------------------------

    df = df[
        df["abstract"]
        .str.split()
        .str.len() >= min_words
    ]

    # --------------------------------------------------------
    # Remove duplicates
    # --------------------------------------------------------

    for col in ["doi", "url", "title"]:
        mask = df[col].notna()

        df_with = df[mask].drop_duplicates(
            subset=[col],
            keep="first"
        )

        df_without = df[~mask]

        df = pd.concat(
            [df_with, df_without],
            ignore_index=True
        )

    df = df.reset_index(drop=True)

    # --------------------------------------------------------
    # Final column ordering
    # --------------------------------------------------------

    ordered_cols = [

        "search_term",

        "doi",
        "url",

        "title",
        "authors",
        "abstract",

        "primary_domain",
        "primary_field",
        "primary_subfield",
        "primary_topic",

        "domains",
        "fields",
        "subfields",
        "topics",

    ]

    remaining = [
        c for c in df.columns
        if c not in ordered_cols
    ]

    return df[
        ordered_cols + remaining
    ]

In [14]:
search_terms = [

    # --------------------------------------------------
    # Computer Science / AI / Data Science
    # --------------------------------------------------
    "machine learning",
    "deep learning",
    "natural language processing",
    "computer vision",
    "reinforcement learning",
    "graph neural networks",
    "data mining",
    "artificial intelligence ethics",

    # --------------------------------------------------
    # Mathematics / Statistics
    # --------------------------------------------------
    "probability theory",
    "statistical inference",
    "Bayesian statistics",
    "stochastic processes",
    "numerical methods",
    "optimization algorithms",
    "linear algebra applications",

    # --------------------------------------------------
    # Physics
    # --------------------------------------------------
    "quantum mechanics",
    "general relativity",
    "particle physics",
    "astrophysics",
    "cosmology",
    "condensed matter physics",

    # --------------------------------------------------
    # Biology / Medicine
    # --------------------------------------------------
    "genomics",
    "bioinformatics",
    "epidemiology",
    "neuroscience",
    "cancer research",
    "medical imaging",
    "public health analytics",

    # --------------------------------------------------
    # Engineering
    # --------------------------------------------------
    "robotics",
    "control systems",
    "signal processing",
    "wireless communication",
    "renewable energy systems",
    "autonomous vehicles",

    # --------------------------------------------------
    # Economics / Finance
    # --------------------------------------------------
    "financial modeling",
    "economic forecasting",
    "market microstructure",
    "behavioral economics",
    "risk management",

    # --------------------------------------------------
    # Social Sciences
    # --------------------------------------------------
    "political polarization",
    "social network analysis",
    "religious switching",
    "education policy",
    "urban sociology",
    "public opinion",

    # --------------------------------------------------
    # Psychology / Cognitive Science
    # --------------------------------------------------
    "cognitive psychology",
    "decision making",
    "memory formation",
    "language acquisition",
    "human perception",

    # --------------------------------------------------
    # Humanities
    # --------------------------------------------------
    "digital humanities",
    "historical linguistics",
    "philosophy of science",
    "ethics of technology",
    "religious studies",
    "classical literature analysis",

    # --------------------------------------------------
    # Interdisciplinary / Modern Topics
    # --------------------------------------------------
    "climate change modeling",
    "computational biology",
    "AI in healthcare",
    "machine learning for finance",
    "network science",
    "complex systems",
    "human computer interaction",

]

df = run_all_searches(
    search_terms=search_terms,
    limit=50
)

Searching: machine learning (1/63)
  Found 50 results
Searching: deep learning (2/63)
  Found 50 results
Searching: natural language processing (3/63)
  Found 50 results
Searching: computer vision (4/63)
  Found 50 results
Searching: reinforcement learning (5/63)
  Found 50 results
Searching: graph neural networks (6/63)
  Found 50 results
Searching: data mining (7/63)
  Found 50 results
Searching: artificial intelligence ethics (8/63)
  Found 50 results
Searching: probability theory (9/63)
  Found 50 results
Searching: statistical inference (10/63)
  Found 50 results
Searching: Bayesian statistics (11/63)
  Found 50 results
Searching: stochastic processes (12/63)
  Found 50 results
Searching: numerical methods (13/63)
  Found 50 results
Searching: optimization algorithms (14/63)
  Found 50 results
Searching: linear algebra applications (15/63)
  Found 50 results
Searching: quantum mechanics (16/63)
  Found 50 results
Searching: general relativity (17/63)
  Found 50 results
Searching: 

In [15]:
cols = ['primary_domain', 'primary_field', 'primary_subfield', 'primary_topic',
        'domains', 'fields', 'subfields', 'topics']

for c in cols:
    vc = df[c].value_counts()
    vc = vc.to_frame()
    display(vc)
    print()

,count
primary_domain,
Physical Sciences,864
Social Sciences,644
Life Sciences,191
Health Sciences,115


,count
primary_field,
Computer Science,384
Social Sciences,278
Physics and Astronomy,195
Engineering,147
"Biochemistry, Genetics and Molecular Biology",115
Medicine,95
Arts and Humanities,95
"Economics, Econometrics and Finance",87
Decision Sciences,81


,count
primary_subfield,
Artificial Intelligence,198
Sociology and Political Science,98
Molecular Biology,94
Astronomy and Astrophysics,77
Computer Vision and Pattern Recognition,61
...,...
"Renewable Energy, Sustainability and the Environment",1
General Energy,1
Discrete Mathematics and Combinatorics,1


,count
primary_topic,
Cosmology and Gravitation Theories,37
Ethics and Social Impacts of AI,33
Religion and Society Interactions,33
Digital Humanities and Scholarship,31
Artificial Intelligence in Healthcare and Education,30
...,...
Power Quality and Harmonics,1
advanced mathematical theories,1
Graph Theory and Algorithms,1


,count
domains,
Physical Sciences,690
Social Sciences,477
Life Sciences,123
"Physical Sciences, Social Sciences",97
"Social Sciences, Physical Sciences",91
Health Sciences,43
"Physical Sciences, Life Sciences",41
"Physical Sciences, Health Sciences",29
"Social Sciences, Life Sciences",29


,count
fields,
Computer Science,200
Social Sciences,146
Physics and Astronomy,129
"Computer Science, Engineering",71
"Biochemistry, Genetics and Molecular Biology",70
...,...
"Nursing, Health Professions, Medicine",1
"Mathematics, Computer Science, Environmental Science",1
"Social Sciences, Economics, Econometrics and Finance, Health Professions",1


,count
subfields,
Artificial Intelligence,83
Molecular Biology,45
Astronomy and Astrophysics,30
"Astronomy and Astrophysics, Nuclear and High Energy Physics",22
Literature and Literary Theory,20
...,...
"Applied Psychology, Clinical Psychology, General Health Professions",1
"Artificial Intelligence, Epidemiology, Health Information Management",1
"Nutrition and Dietetics, General Health Professions, Public Health, Environmental and Occupational Health",1


,count
topics,
Digital Humanities and Scholarship,17
"Genetics, Bioinformatics, and Biomedical Research",9
Matrix Theory and Algorithms,7
Cognitive Science and Mapping,7
"Neuroscience, Education and Cognitive Function",6
...,...
"Computability, Logic, AI Algorithms, Neural Networks and Applications",1
"Psychology of Moral and Emotional Judgment, Decision-Making and Behavioral Economics, Neural and Behavioral Psychology Studies",1
"Hearing, Cochlea, Tinnitus, Genetics, Hearing Loss and Rehabilitation, Vestibular and auditory disorders",1


In [6]:
df.columns

Index(['search_term', 'doi', 'url', 'title', 'authors', 'abstract',
       'primary_domain', 'primary_field', 'primary_subfield', 'primary_topic',
       'domains', 'fields', 'subfields', 'topics'],
      dtype='object')